# Amazon ML Challenge 2026 — Business Entity Resolution (M1 Pro, MLX-first)

Frozen multilingual embeddings (bge-m3 in pure MLX) + multi-retriever blocking + LightGBM and an MLX Siamese-tower MLP, blended, isotonic-calibrated, and decoded per S1 by expected F0.5.

* Run from `student_resource/` with the **er-mac** kernel. Run `SMOKE=True` first; paste every `>>> PASTE TO CLAUDE` line back.
* Every expensive step caches to `artifacts/` (`artifacts_smoke/` in smoke mode) and is skipped when the cache matches the current parameters; `FORCE=True` recomputes.
* Heavy code lives next to this notebook in `er_common.py`, `er_text.py`, `er_mlx_encoder.py`, `er_embed.py`, `er_block.py`, `er_feats.py`, `er_models.py`, `er_decide.py`.

## Cell 0 — Parameters

In [9]:
SMOKE = True        # I will run SMOKE first, then set False for the full run
FORCE = False
EMB_REPO = 'mlx-community/bge-m3-mlx-fp16'
EMB_BATCH = 64      # I'll update this from the cell 1b benchmark
K_TFIDF = 30; K_DENSE = 30; K_RARE = 10
N_FOLDS = 5
SEED = 42
N_JOBS = None       # None -> performance-core count
BACKEND = 'mlx'     # 'mlx' or 'torch_mps' fallback for the MLP
ONE_TO_ONE = None   # None -> decided automatically in cell 3; True/False to override
SAME_COUNTRY = None # None -> decided automatically in cell 3

# ---- additional knobs (defaults are fine; see RUN.md) ----
USE_ST_MPS = False           # True -> cell 6 embeds with sentence-transformers BAAI/bge-m3 on MPS fp16 (set if cell 1a FAILs)
EMB_ST_REF = 'BAAI/bge-m3'   # reference model for the parity check and the MPS fallback
EMB_POOLING = 'cls'          # bge-m3 dense vector = CLS token + L2 norm (NOT mean pooling)
EMB_MAX_LEN = 96             # tokens; names/addresses are short
EMB_DIM = 256                # PCA dim for storage + search (None keeps 1024: ~12M test rows x 1024 x fp16 = 24 GB per field)
EMB_FIELDS = ('name', 'addr', 'combo')  # drop 'combo' to cut embedding time ~45%; retrieval then uses [name;addr]
K_COMBO = 20                 # extra retriever: word TF-IDF on core name + address
TFIDF_MAX_DF = 0.005; TFIDF_MAX_DF_MIN = 500   # drop n-grams in > max(0.5% of docs, 500) records
PRUNE_CHAR = 24; PRUNE_WORD = 12               # query-side pruning to the most informative grams/tokens
RARE_MAX_DF = 30             # 'rare' token = appears in <= 30 S2/S3 records of the group
TRAIN_S1_N = 150_000         # full run: train S1s covered by region sampling (memory-bound; see RUN.md)
LGB_ROUNDS = 3000; LGB_ES = 100
MLP_EPOCHS = 8; MLP_BS = 1024; MLP_COMPARE_PW = True
MC_DRAWS = 1000
FEAT_CHUNK = 1_000_000

In [10]:
# setup: paths, env, imports, seeds
import os, sys, gc, json, time, platform, subprocess
from pathlib import Path

def _find_root():
    """student_resource/ = the folder holding dataset/train and er_common.py; works from any launch folder."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        for cand in (p, p / 'student_resource'):
            if (cand / 'dataset' / 'train').is_dir() and (cand / 'er_common.py').is_file():
                return cand
    raise FileNotFoundError(f'could not find student_resource/ (with dataset/train and er_common.py) from cwd={here}')

ROOT = _find_root()
os.chdir(ROOT)   # all relative paths below (dataset/, utils/, artifacts/) are relative to student_resource/
os.environ.setdefault('HF_HOME', str(ROOT / 'hf_cache'))   # reuse the already-downloaded weights
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('python:', sys.executable, platform.python_version(), platform.machine())
try:
    import mlx.core as mx
except ImportError as e:
    raise ImportError(f'{e}. This kernel is not the er-mac venv (python={sys.executable}). '
                      'Run `bash setup_mac.sh`, then Kernel -> Change Kernel -> er-mac, and rerun from cell 0.') from None
import numpy as np
import pandas as pd
from er_common import (section, check, paste, stage_dir, mark_done, read_tsv, parse_ground_truth,
                       region_sample_train, random_sample_train, smoke_sample_test, build_records, SECTION_LOG)
import er_text, er_embed, er_block, er_feats, er_models, er_decide
np.random.seed(SEED)
mx.random.seed(SEED)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 60)
check('.venv_mlx' in sys.executable, f'kernel python is the er-mac venv ({sys.executable})')
print('setup ok', ROOT)

python: /Users/sudarshansudhakar/Desktop/RESEARCH/Amazon ML 2026/student_resource/.venv_mlx/bin/python 3.12.2 arm64
✅ PASS: kernel python is the er-mac venv (/Users/sudarshansudhakar/Desktop/RESEARCH/Amazon ML 2026/student_resource/.venv_mlx/bin/python)
setup ok /Users/sudarshansudhakar/Desktop/RESEARCH/Amazon ML 2026/student_resource


## Cell 1 — Environment check

In [11]:
with section('1 environment'):
    sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
    arch = platform.machine()
    chip = sh('sysctl -n machdep.cpu.brand_string')
    pcores = int(sh('sysctl -n hw.perflevel0.physicalcpu') or os.cpu_count())
    ram_gb = int(sh('sysctl -n hw.memsize')) / 2**30
    # torch is checked in a SUBPROCESS: importing torch into this kernel loads a second OpenMP runtime
    # (torch/lib/libomp.dylib) next to LightGBM's Homebrew libomp, which segfaults LightGBM in cell 10.
    tq = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.backends.mps.is_available())'],
                        capture_output=True, text=True)
    torch_ver, mps = (tq.stdout.split() + ['?', 'False'])[:2] if tq.returncode == 0 else ('import-failed', 'False')
    mps = mps == 'True'
    dev = mx.default_device()
    if N_JOBS is None:
        N_JOBS = pcores
    print(f'arch={arch}  chip={chip}  P-cores={pcores}  RAM={ram_gb:.0f} GB')
    print(f'mlx={mx.__version__}  default_device={dev}  torch={torch_ver}  mps={mps}  N_JOBS={N_JOBS}  python={platform.python_version()}')
    assert arch == 'arm64', 'x86_64 interpreter (Rosetta): rebuild the venv with native arm64 Python (setup_mac.sh)'
    ok = check(arch == 'arm64' and 'gpu' in str(dev).lower(), f'native arm64, MLX default device is the GPU ({dev})')
    check(mps, 'torch MPS available (only needed for fallbacks)')
    check('torch' not in sys.modules, 'torch is not loaded in this kernel (avoids the OpenMP clash with LightGBM)')
    paste(f'cell1 arch={arch} chip={chip} pcores={pcores} ram={ram_gb:.0f}GB mlx={mx.__version__} device={dev} torch={torch_ver} mps={mps} N_JOBS={N_JOBS} pass={ok}')

── 1 environment ── start (RSS 0.18 GB)
arch=arm64  chip=Apple M1 Pro  P-cores=8  RAM=16 GB
mlx=0.32.2  default_device=Device(gpu, 0)  torch=2.14.0  mps=True  N_JOBS=8  python=3.12.2
✅ PASS: native arm64, MLX default device is the GPU (Device(gpu, 0))
✅ PASS: torch MPS available (only needed for fallbacks)
✅ PASS: torch is not loaded in this kernel (avoids the OpenMP clash with LightGBM)
>>> PASTE TO CLAUDE: cell1 arch=arm64 chip=Apple M1 Pro pcores=8 ram=16GB mlx=0.32.2 device=Device(gpu, 0) torch=2.14.0 mps=True N_JOBS=8 pass=True
── 1 environment ── 1.1s | RSS 0.18 GB | peak RSS 0.22 GB


## Cell 1a — Embedding parity check (MLX vs sentence-transformers)

BAAI/bge-m3's dense embedding is the **CLS token + L2 normalisation**. The MLX path must reproduce sentence-transformers `BAAI/bge-m3` on CPU with mean cosine ≥ 0.999 on ~50 English / Indian / French / Devanagari / Gurmukhi strings.

`mlx-embeddings` is **GPL-3.0**, which the rules forbid, so the XLM-R encoder is implemented directly in `mlx.nn` (`er_mlx_encoder.py`) and loads the same `mlx-community/bge-m3-mlx-fp16` safetensors.

In [12]:
with section('1a parity'):
    # Runs in a subprocess so sentence-transformers/torch never load into this kernel
    # (torch's bundled libomp + LightGBM's libomp in one process segfaults LightGBM on macOS).
    code = ('import json, sys; sys.path.insert(0, "."); import er_embed; '
            f'print("PARITY_JSON" + json.dumps(er_embed.parity_check({EMB_REPO!r}, {EMB_ST_REF!r}, {EMB_POOLING!r}, {EMB_MAX_LEN})))')
    res = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, cwd=ROOT, env=os.environ.copy())
    line = next((l for l in res.stdout.splitlines() if l.startswith('PARITY_JSON')), None)
    if line is None:
        print(res.stdout[-3000:]); print(res.stderr[-5000:])
        raise RuntimeError('parity subprocess failed; see the output above')
    par = json.loads(line[len('PARITY_JSON'):])
    print(json.dumps(par, indent=1, ensure_ascii=False))
    m_cos, min_cos, wrong_cos, st_pool, worst = par['mean_cos'], par['min_cos'], par['wrong_pool_mean_cos'], par['st_pooling'], par['worst']
    PARITY_OK = check(m_cos >= 0.999, f'MLX fp16 {EMB_POOLING}+L2 vs sentence-transformers CPU fp32: mean cos {m_cos:.5f}, min {min_cos:.5f}')
    if not PARITY_OK:
        print('DIAGNOSIS')
        print(f'  sentence-transformers pooling module: {st_pool}')
        print(f'  the other pooling gives mean cos {wrong_cos:.4f}' + ('   <- pooling mismatch' if wrong_cos > m_cos else ''))
        print(f'  worst string: {worst!r}')
        print('  FIX: set USE_ST_MPS = True in cell 0 (cell 6 then embeds with sentence-transformers on MPS, fp16) and rerun from cell 0.')
        print('  NOTE: USE_ST_MPS loads torch into this kernel; restart the kernel after cell 6 before running cell 10 (LightGBM).')
    check('torch' not in sys.modules, 'torch still not loaded in this kernel')
    del par; gc.collect()
    paste(f'cell1a mean_cos={m_cos:.5f} min_cos={min_cos:.5f} other_pooling={wrong_cos:.4f} st_pooling={st_pool} pass={PARITY_OK}')

── 1a parity ── start (RSS 0.18 GB)
{
 "n": 50,
 "mean_cos": 0.9999992251396179,
 "min_cos": 0.9999601244926453,
 "worst": "Jain Systems Pvt Ltd",
 "wrong_pool_mean_cos": 0.7745643854141235,
 "st_pooling": "Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})",
 "dim": 1024
}
✅ PASS: MLX fp16 cls+L2 vs sentence-transformers CPU fp32: mean cos 1.00000, min 0.99996
✅ PASS: torch still not loaded in this kernel
>>> PASTE TO CLAUDE: cell1a mean_cos=1.00000 min_cos=0.99996 other_pooling=0.7746 st_pooling=Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True}) pass=True
── 1a parity ── 22.5s | RSS 0.19 GB | peak RSS 0.22 GB


## Cell 1b — Embedding benchmark

Throughput on real names / addresses / combined texts at batch 32 / 64 / 128 (length-sorted batches), and a projection of the full-run embedding time.

In [13]:
with section('1b benchmark'):
    smp = pd.concat([read_tsv('dataset/test/test_source2.tsv', nrows=3000), read_tsv('dataset/test/test_source1.tsv', nrows=1500),
                     read_tsv('dataset/train/train_source3.tsv', nrows=1500)], ignore_index=True)
    smp = smp.sample(frac=1.0, random_state=SEED)
    nm = [er_text.emb_text(x) for x in smp['business_name']]
    ad = [er_text.emb_text(x) for x in smp['business_address']]
    bench_texts = {'name': nm, 'addr': ad, 'combo': [er_text.emb_text(a + ' | ' + b) for a, b in zip(nm, ad)]}
    embr = er_embed.Embedder(EMB_REPO, EMB_POOLING, EMB_MAX_LEN, '', USE_ST_MPS, EMB_ST_REF)
    bench = er_embed.benchmark(embr, bench_texts, (32, 64, 128), n=2048)
    embr.close(); del embr; gc.collect()
    print('records/sec per field (rows) and batch size (columns):')
    print(pd.DataFrame(bench).T.round(0).to_string())
    per_rec = {bs: 1.0 / sum(1.0 / bench[f][bs] for f in EMB_FIELDS) for bs in (32, 64, 128)}
    best_bs = max(per_rec, key=per_rec.get)
    best_rate = per_rec[best_bs]
    n_lines = lambda p: sum(1 for _ in open(p, 'rb')) - 1
    n_test = sum(n_lines(f'dataset/test/test_source{i}.tsv') for i in (1, 2, 3))
    n_train_est = int(TRAIN_S1_N * 6.5)   # S1 + ~3.5 matches + regional competitors (cell 2 prints the real number)
    hrs = (n_test + n_train_est) / best_rate / 3600
    hrs_nc = (n_test + n_train_est) * sum(1.0 / bench[f][best_bs] for f in ('name', 'addr')) / 3600
    print(f'best batch {best_bs}: {best_rate:,.0f} records/s for fields {EMB_FIELDS}')
    print(f'full run embeds ~{n_test:,} test + ~{n_train_est:,} train records -> {hrs:.1f} h ({hrs_nc:.1f} h with EMB_FIELDS=(name, addr))')
    check(best_rate > 0, 'benchmark ran')
    paste(f'cell1b best batch={best_bs}, rec/s={best_rate:.0f}, projected={hrs:.1f}h (no-combo {hrs_nc:.1f}h), per-field@best name={bench["name"][best_bs]:.0f} addr={bench["addr"][best_bs]:.0f} combo={bench["combo"][best_bs]:.0f}')

── 1b benchmark ── start (RSS 0.19 GB)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

records/sec per field (rows) and batch size (columns):
         32     64     128
name   409.0  460.0  476.0
addr   241.0  254.0  239.0
combo  144.0  135.0  155.0
best batch 128: 79 records/s for fields ('name', 'addr', 'combo')
full run embeds ~11,702,133 test + ~975,000 train records -> 44.8 h (22.1 h with EMB_FIELDS=(name, addr))
✅ PASS: benchmark ran
>>> PASTE TO CLAUDE: cell1b best batch=128, rec/s=79, projected=44.8h (no-combo 22.1h), per-field@best name=476 addr=239 combo=155
── 1b benchmark ── 95.0s | RSS 0.28 GB | peak RSS 1.60 GB


## Cell 1c — Smoke overrides and output folders

In [14]:
SMOKE_TRAIN_S1, SMOKE_TEST_S1 = 300, 200
if SMOKE:
    K_TFIDF, K_DENSE, K_RARE, K_COMBO = 10, 10, 5, 8
    N_FOLDS = 3; LGB_ROUNDS = 150; LGB_ES = 30; MC_DRAWS = 300; FEAT_CHUNK = 200_000
    MLP_EPOCHS = 4; MLP_BS = 128   # ~4k smoke pairs: batch 1024 would give only 3 optimizer steps per epoch
ART = ROOT / ('artifacts_smoke' if SMOKE else 'artifacts')
OUT = ROOT / ('output_smoke' if SMOKE else 'output')
REP = ROOT / 'reports'
for d in (ART, OUT, REP):
    d.mkdir(exist_ok=True)
CFG_B = dict(K_TFIDF=K_TFIDF, K_COMBO=K_COMBO, K_DENSE=K_DENSE, K_RARE=K_RARE, TFIDF_MAX_DF=TFIDF_MAX_DF,
             TFIDF_MAX_DF_MIN=TFIDF_MAX_DF_MIN, PRUNE_CHAR=PRUNE_CHAR, PRUNE_WORD=PRUNE_WORD, RARE_MAX_DF=RARE_MAX_DF)
print(f'SMOKE={SMOKE}  ART={ART.name}  OUT={OUT.name}  folds={N_FOLDS}  lgb_rounds={LGB_ROUNDS}  mlp_epochs={MLP_EPOCHS}  mlp_bs={MLP_BS}')
print('blocking config', CFG_B)

SMOKE=True  ART=artifacts_smoke  OUT=output_smoke  folds=3  lgb_rounds=150  mlp_epochs=4  mlp_bs=128
blocking config {'K_TFIDF': 10, 'K_COMBO': 8, 'K_DENSE': 10, 'K_RARE': 5, 'TFIDF_MAX_DF': 0.005, 'TFIDF_MAX_DF_MIN': 500, 'PRUNE_CHAR': 24, 'PRUNE_WORD': 12, 'RARE_MAX_DF': 30}


## Cell 2 — Load data and build the working sample

* **Smoke:** ~300 train S1 (every country) + all their true matches + 3× random other S2/S3; ~200 test S1 (every country) + test S2/S3 sharing a distinctive name token with them + 3,000 random per country.
* **Full:** train is **region-sampled** (whole cities/states chosen from frequent address segments until ~`TRAIN_S1_N` S1s are covered, keeping every S2/S3 record of those regions) so candidate density matches the full test pool; test is used in full.

In [15]:
with section('2 load'):
    tr1, tr2, tr3 = (read_tsv(f'dataset/train/train_source{i}.tsv') for i in (1, 2, 3))
    gt = read_tsv('dataset/train/train_ground_truth.tsv')
    te1, te2, te3 = (read_tsv(f'dataset/test/test_source{i}.tsv') for i in (1, 2, 3))
    for nm_, df_ in [('train_s1', tr1), ('train_s2', tr2), ('train_s3', tr3), ('train_gt', gt), ('test_s1', te1), ('test_s2', te2), ('test_s3', te3)]:
        print(f'  {nm_:9s} {len(df_):>10,} rows  columns={list(df_.columns)}')
    pos_all, n_true_all = parse_ground_truth(gt)
    tr23 = pd.concat([tr2, tr3], ignore_index=True)
    te23 = pd.concat([te2, te3], ignore_index=True)
    TEST_S1_IDS = te1['entity_id'].tolist()
    # v2: records are sorted by entity_id inside build_records (row order independent of how the sample is drawn)
    SAMPLE_P = {'SMOKE': SMOKE, 'TRAIN_S1_N': TRAIN_S1_N, 'SMOKE_TRAIN_S1': SMOKE_TRAIN_S1, 'SMOKE_TEST_S1': SMOKE_TEST_S1, 'SEED': SEED, 'v': 2}
    sd, fresh = stage_dir(ART, 'sample', SAMPLE_P, FORCE)
    if fresh:
        keep = {s: set(pd.read_parquet(sd / f'{s}.parquet')['entity_id']) for s in ('train', 'test')}
        s1_tr, s23_tr = tr1[tr1['entity_id'].isin(keep['train'])], tr23[tr23['entity_id'].isin(keep['train'])]
        s1_te, s23_te = te1[te1['entity_id'].isin(keep['test'])], te23[te23['entity_id'].isin(keep['test'])]
        print('  sample loaded from cache')
    else:
        if SMOKE:
            s1_tr, s23_tr = random_sample_train(tr1, tr23, pos_all, SMOKE_TRAIN_S1, SEED)
            s1_te, s23_te = smoke_sample_test(te1, te23, SMOKE_TEST_S1, SEED)
        else:
            s1_tr, s23_tr, n_regions = region_sample_train(tr1, tr23, pos_all, TRAIN_S1_N, SEED)
            print(f'  region sampling chose {n_regions} address segments')
            s1_te, s23_te = te1, te23
        for s, (a, b) in {'train': (s1_tr, s23_tr), 'test': (s1_te, s23_te)}.items():
            pd.DataFrame({'entity_id': np.concatenate([a['entity_id'].to_numpy(dtype=object), b['entity_id'].to_numpy(dtype=object)])}).to_parquet(sd / f'{s}.parquet')
        mark_done(sd)

    def split_src(s1, s23):
        e = s23['entity_id']
        return build_records(s1, s23[e.str.startswith('S2-')], s23[e.str.startswith('S3-')])

    rec = {'train': split_src(s1_tr, s23_tr), 'test': split_src(s1_te, s23_te)}
    del s1_tr, s23_tr, s1_te, s23_te
    REC_IDS = {s: rec[s]['entity_id'].to_numpy(dtype=object).copy() for s in rec}   # row order every cache is indexed by

    # positives of the sampled train S1s -> global row ids of rec['train']
    rtr = rec['train']
    pos = pos_all[pos_all['s1'].isin(set(rtr.loc[rtr['src'] == 1, 'entity_id']))]
    ix = pd.Index(rtr['entity_id'])
    pq, pt = ix.get_indexer(pos['s1']), ix.get_indexer(pos['t'])
    assert (pq >= 0).all() and (pt >= 0).all(), 'a true match of a sampled S1 is missing from the sample'
    NR_TR = len(rtr)
    POS_KEYS = np.sort(pq.astype(np.int64) * NR_TR + pt)
    N_TRUE = np.bincount(pq, minlength=NR_TR).astype(np.int64)
    for s, r in rec.items():
        print(f'  {s} sample (rows = country, columns = source):')
        print(pd.crosstab(r['country'], r['src']).to_string())
    s1_mask = rtr['src'].to_numpy() == 1
    samp_sing = float((N_TRUE[s1_mask] == 0).mean())
    ok_c = check(set(rec['test']['country']) == set(te1['country']), f'every test country is in the test sample: {sorted(set(rec["test"]["country"]))}')
    ok_p = check(len(POS_KEYS) > 0, f'{len(POS_KEYS):,} true pairs inside the train sample; sample singleton rate {samp_sing:.3f}')
    paste(f'cell2 train_rec={len(rtr):,} train_S1={int(s1_mask.sum()):,} true_pairs={len(POS_KEYS):,} test_rec={len(rec["test"]):,} test_S1_all={len(TEST_S1_IDS):,} countries_ok={ok_c}')

── 2 load ── start (RSS 0.28 GB)
  train_s1   2,206,821 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  train_s2   5,034,616 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  train_s3   5,285,603 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  train_gt   2,206,821 rows  columns=['source1_entity_id', 'matched_entity_ids']
  test_s1    1,732,544 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  test_s2    4,887,273 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  test_s3    5,082,316 rows  columns=['entity_id', 'business_name', 'business_address', 'country']
  sample loaded from cache
  train sample (rows = country, columns = source):
src        1     2     3
country                 
India    120   850   879
US       180  1278  1289
  test sample (rows = country, columns = source):
src       1      2      3
country                  
France   30 

## Cell 3 — EDA and automatic design decisions

(a) Is any S2/S3 ID matched to more than one S1? If not → `ONE_TO_ONE=True`.  (b) Do all matched pairs share the country string? If so → `SAME_COUNTRY=True` (blocking groups by the country *string*; country is never a model feature).

In [16]:
with section('3 EDA'):
    for nm_, df_ in [('train_s1', tr1), ('train_s2', tr2), ('train_s3', tr3), ('test_s1', te1), ('test_s2', te2), ('test_s3', te3)]:
        print(f'  {nm_:9s} {len(df_):>10,}  {df_["country"].value_counts().to_dict()}')
    print('sample rows (test, 2 per country per source):')
    print(te1.groupby('country').head(2).to_string(index=False))
    print(te23.groupby('country').head(2).to_string(index=False))
    sing_rate = float((n_true_all == 0).mean())
    print(f'train singleton rate {sing_rate:.4f}; matches per S1 (10 = 10+):')
    print(n_true_all.clip(upper=10).value_counts().sort_index().to_string())
    is2 = pos_all['t'].str.startswith('S2-')
    g2 = is2.groupby(pos_all['s1']).agg(['any', 'all'])
    n_both, n_s2only, n_s3only = int((g2['any'] & ~g2['all']).sum()), int(g2['all'].sum()), int((~g2['any']).sum())
    print(f'non-singleton S1: both={n_both:,}  S2-only={n_s2only:,}  S3-only={n_s3only:,}')

    rng = np.random.default_rng(SEED)
    pres = []
    for nm_, df_ in [('train_s1', tr1), ('train_s23', tr23), ('test_s1', te1), ('test_s23', te23)]:
        smp = df_.iloc[rng.choice(len(df_), min(len(df_), 200_000), replace=False)]
        a = smp['business_address']
        pres.append(pd.DataFrame({'file': nm_, 'country': smp['country'].to_numpy(),
                                  'postal_5_6': a.str.contains(r'\b\d{5,6}\b').to_numpy(),
                                  'postal_3+3': a.str.contains(r'\b\d{3} \d{3}\b').to_numpy(),
                                  'postal_4': a.str.contains(r'\b\d{4}\b').to_numpy(),
                                  'lead_street_no': a.str.match(r'\s*\d+').to_numpy(),
                                  'any_digit': a.str.contains(r'\d').to_numpy(),
                                  'addr_empty': (a.str.strip() == '').to_numpy()}))
    pres = pd.concat(pres).groupby(['file', 'country']).mean().round(3)
    print('postal-code / street-number presence (fraction of records):')
    print(pres.to_string())

    # (a) one-to-one?
    n_multi = int(pos_all['t'].duplicated().sum())
    # (b) same country string?
    c1 = pos_all['s1'].map(tr1.set_index('entity_id')['country'])
    ct = pos_all['t'].map(tr23.set_index('entity_id')['country'])
    n_same = int((c1.to_numpy() == ct.to_numpy()).sum())
    same_frac = n_same / max(len(pos_all), 1)

    rtr = rec['train']
    cols = ['entity_id', 'business_name', 'business_address', 'country']
    def show_pairs(iq, it, title):
        a = rtr.iloc[iq][cols].reset_index(drop=True).add_prefix('S1_')
        b = rtr.iloc[it][cols[:3]].reset_index(drop=True).add_prefix('T_')
        print(title); print(pd.concat([a, b], axis=1).to_string(index=False))
    sel = np.random.default_rng(SEED).choice(len(POS_KEYS), min(10, len(POS_KEYS)), replace=False)
    show_pairs(POS_KEYS[sel] // NR_TR, POS_KEYS[sel] % NR_TR, '10 matched pairs:')
    key = rtr['business_name'].str.casefold().str.replace(r'[^\w]+', ' ', regex=True).str.strip()
    kd = pd.DataFrame({'i': np.arange(len(rtr)), 'k': key.to_numpy(dtype=object), 'src': rtr['src'].to_numpy(), 'c': rtr['country'].to_numpy(dtype=object)})
    mm = kd[kd['src'] == 1].merge(kd[kd['src'] != 1], on=['k', 'c'], suffixes=('_q', '_t'))
    mm = mm[~np.isin(mm['i_q'].to_numpy(np.int64) * NR_TR + mm['i_t'].to_numpy(np.int64), POS_KEYS)]
    print(f'identical-name S1 x S2/S3 pairs in the train sample that are NOT matches: {len(mm):,}')
    if len(mm):
        hs = mm.sample(min(10, len(mm)), random_state=SEED)
        show_pairs(hs['i_q'].to_numpy(), hs['i_t'].to_numpy(), '10 hard non-matches (same name, different business):')

    if ONE_TO_ONE is None:
        ONE_TO_ONE = bool(n_multi == 0)
    if SAME_COUNTRY is None:
        SAME_COUNTRY = bool(n_same == len(pos_all))
    print('DECISIONS')
    print(f'  (a) S2/S3 IDs under >1 S1: {n_multi:,} of {len(pos_all):,} matched IDs  ->  ONE_TO_ONE={ONE_TO_ONE}')
    print(f'  (b) matched pairs with the same country string: {n_same:,}/{len(pos_all):,} ({same_frac:.6f})  ->  SAME_COUNTRY={SAME_COUNTRY}')
    check(True, 'EDA done')
    paste(f'cell3 singleton={sing_rate:.4f} mean_matches_nonsing={n_true_all[n_true_all > 0].mean():.2f} both/S2only/S3only={n_both}/{n_s2only}/{n_s3only} multi_S1_ids={n_multi} same_country={same_frac:.6f} ONE_TO_ONE={ONE_TO_ONE} SAME_COUNTRY={SAME_COUNTRY} hard_nonmatch_same_name={len(mm)}')
    del tr1, tr2, tr3, tr23, te1, te2, te3, te23, gt, c1, ct, pos_all, kd, mm
    gc.collect()

── 3 EDA ── start (RSS 5.49 GB)
  train_s1   2,206,821  {'US': 1323633, 'India': 883188}
  train_s2   5,034,616  {'US': 3016817, 'India': 2017799}
  train_s3   5,285,603  {'US': 3170056, 'India': 2115547}
  test_s1    1,732,544  {'India': 809986, 'US': 663106, 'France': 259452}
  test_s2    4,887,273  {'India': 2312565, 'US': 1871330, 'France': 703378}
  test_s3    5,082,316  {'India': 2405000, 'US': 1945701, 'France': 731615}
sample rows (test, 2 per country per source):
   entity_id        business_name                                                            business_address country
S1-714132312      Zephay Labs Inc                                                 2621 Cotten Road, Tyler, TX      US
S1-106407869 Vision Partners Corp                                      IA, Iowa City, 1064 Newton Rd, Unit 11      US
S1-156285671        << Team Ecole 175 Boulevard du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine  France
S1-689823050  Red Perfect Trading           Mirzapu

## Cell 4 — Normalisation

NFKD → Brahmic-script transliteration (one hand-written table for all nine ISCII-aligned Indic blocks) → accent strip → casefold → `&`→and → punctuation/whitespace. Legal suffix split into its own field (English, Indian, French forms); address abbreviations expanded (EN/IN/FR); digits, postal-like token, street number and landmark flag extracted; sorted-token name. No country-specific regexes.

In [17]:
with section('4 normalise'):
    nd, fresh = stage_dir(ART, 'norm', {'sample': SAMPLE_P, 'n_train': len(rec['train']), 'n_test': len(rec['test']), 'v': 2}, FORCE)
    for split in ('train', 'test'):
        p = nd / f'{split}.parquet'
        if not (fresh and p.exists()):
            t0 = time.time()
            er_text.normalize_records(rec[split], N_JOBS).to_parquet(p, index=False)
            print(f'  {split}: normalised {len(rec[split]):,} records in {time.time() - t0:,.0f}s')
        rec[split] = pd.read_parquet(p, dtype_backend='pyarrow')
        # every cache (embeddings, pairs, features) and POS_KEYS / N_TRUE is indexed by this row order
        same = len(rec[split]) == len(REC_IDS[split]) and (rec[split]['entity_id'].to_numpy(dtype=object) == REC_IDS[split]).all()
        assert same, f'{split}: cached normalised table is in a different row order than cell 2 -> rerun with FORCE=True'
    mark_done(nd)
    check(True, 'row order of the normalised tables matches cell 2 (caches and labels aligned)')
    r = rec['test']
    show = ['business_name', 'name_n', 'name_core', 'legal', 'business_address', 'addr_n', 'postal', 'stnum', 'landmark']
    brahmic = r[r['business_name'].str.contains('[ऀ-ൿ]', regex=True)].head(4)
    print(pd.concat([r.groupby('country').head(3), brahmic])[show].to_string(index=False))
    stats = pd.DataFrame({'split_country': np.concatenate([(s + '/' + rec[s]['country'].astype(str)).to_numpy(dtype=object) for s in rec]),
                          'core_empty': np.concatenate([(rec[s]['name_core'] == '').to_numpy(bool) for s in rec]),
                          'has_legal': np.concatenate([(rec[s]['legal'] != '').to_numpy(bool) for s in rec]),
                          'has_postal': np.concatenate([(rec[s]['postal'] != '').to_numpy(bool) for s in rec]),
                          'has_stnum': np.concatenate([(rec[s]['stnum'] != '').to_numpy(bool) for s in rec]),
                          'landmark': np.concatenate([rec[s]['landmark'].to_numpy() for s in rec]).astype(bool),
                          'non_ascii_left': np.concatenate([rec[s]['name_n'].str.contains(r'[^\x00-\x7f]', regex=True).to_numpy(bool) for s in rec])})
    st = stats.groupby('split_country').mean().round(3)
    print(st.to_string())
    ok = check(st['core_empty'].max() < 0.01, f'core names non-empty (max empty rate {st["core_empty"].max():.4f})')
    paste('cell4 ' + '; '.join(f'{i}: legal={row.has_legal:.2f} postal={row.has_postal:.2f} stnum={row.has_stnum:.2f} lm={row.landmark:.2f} nonascii={row.non_ascii_left:.3f}' for i, row in st.iterrows()) + f' pass={ok}')

── 4 normalise ── start (RSS 6.01 GB)
  train: normalised 4,596 records in 1s
  test: normalised 49,163 records in 2s
✅ PASS: row order of the normalised tables matches cell 2 (caches and labels aligned)
                          business_name                                  name_n                               name_core   legal                                                                                                                      business_address                                                                                                              addr_n postal stnum  landmark
             Galav Aqua Private Limited              galav aqua private limited                              galav aqua ltd pvt                                                                                              2/434 H Sry 2615, Lucknow, Uttar Pradesh                                                                                2 434 sry 2615 lucknow uttar pradesh   2615     2     

## Cell 5 — Metric: macro F0.5

In [18]:
with section('5 metric'):
    f05 = er_decide.f05
    ex = f05({'a', 'b', 'c'}, {'a', 'c'})
    assert round(ex, 3) == 0.714, ex
    assert f05(set(), set()) == 1.0
    assert f05({'x'}, set()) == 0.0
    assert f05(set(), {'a'}) == 0.0
    two = er_decide.macro_f05({'s1': {'a', 'b', 'c'}}, {'s1': {'a', 'c'}, 's2': set()})
    assert abs(two - (ex + 1.0) / 2) < 1e-9, two
    pc = er_decide.per_country_f05({'s1': {'a', 'c'}}, {'s1': {'a', 'c'}, 's2': {'z'}}, {'s1': 'X', 's2': 'Y'})
    assert pc == {'X': 1.0, 'Y': 0.0}, pc
    # the vectorised version (used everywhere downstream) must agree with the dict version
    rng = np.random.default_rng(SEED)
    nq, npair = 300, 3000
    qv = rng.integers(0, nq, npair); tv = np.arange(npair)
    yv = (rng.random(npair) < 0.3).astype(np.int8); sv = rng.random(npair) < 0.35
    missed = rng.integers(0, 2, nq)
    ntv = np.bincount(qv[yv == 1], minlength=nq) + missed
    truth = {s: {int(t) for t in tv[(qv == s) & (yv == 1)]} | {f'miss{s}' for _ in range(missed[s])} for s in range(nq)}
    pred = {s: {int(t) for t in tv[(qv == s) & sv]} for s in range(nq)}
    d1 = er_decide.macro_f05(pred, truth)
    d2 = er_decide.f05_vec(qv, sv, yv, ntv, np.arange(nq)).mean()
    assert abs(d1 - d2) < 1e-9, (d1, d2)
    check(True, f'metric asserts: example={ex:.3f} (0.714), singleton+empty=1, singleton+pred=0, nonsingleton+empty=0, vectorised==dict ({d1:.6f})')
    paste(f'cell5 metric PASS example={ex:.4f}')

── 5 metric ── start (RSS 6.04 GB)
✅ PASS: metric asserts: example=0.714 (0.714), singleton+empty=1, singleton+pred=0, nonsingleton+empty=0, vectorised==dict (0.320959)
>>> PASTE TO CLAUDE: cell5 metric PASS example=0.7143
── 5 metric ── 0.0s | RSS 6.04 GB | peak RSS 6.91 GB


## Cell 6 — Embeddings with MLX (frozen bge-m3, fp16, CLS + L2 exactly as in cell 1a)

Encodes **name**, **address** and **`name | address`** for every train-sample and test record; `USE_ST_MPS=True` switches to sentence-transformers on MPS. Texts are length-sorted before batching and restored to their original order afterwards, with `mx.eval` called per batch. Vectors are projected with a train-fitted PCA to `EMB_DIM`, re-L2-normalised, and written chunk by chunk to float16 `.npy` memmaps whose row index is the record table. A resume marker lets an interrupted run continue where it stopped.

In [19]:
assert {'name', 'addr'} <= set(EMB_FIELDS), 'name and addr embeddings are required (MLP towers + features)'

def field_texts(r, field):
    nm = r['business_name'].astype(str).tolist()
    if field == 'name':
        return [er_text.emb_text(x) for x in nm]
    ad = r['business_address'].astype(str).tolist()
    if field == 'addr':
        return [er_text.emb_text(x) for x in ad]
    return [er_text.emb_text(a + ' | ' + b) for a, b in zip(nm, ad)]

EMB_P = {'sample': SAMPLE_P, 'repo': EMB_REPO, 'pool': EMB_POOLING, 'max_len': EMB_MAX_LEN, 'dim': EMB_DIM,
         'fields': list(EMB_FIELDS), 'st_mps': USE_ST_MPS, 'v': 2}
with section('6 embeddings'):
    ed, fresh = stage_dir(ART, 'emb', EMB_P, FORCE)
    rng = np.random.default_rng(SEED)
    EMB_RATES = {}
    if not fresh:
        embr = er_embed.Embedder(EMB_REPO, EMB_POOLING, EMB_MAX_LEN, '', USE_ST_MPS, EMB_ST_REF)
        for field in EMB_FIELDS:
            pca_p = ed / f'pca_{field}.npz'
            if EMB_DIM and not pca_p.exists():
                texts = field_texts(rec['train'], field)
                idx = rng.choice(len(texts), min(len(texts), 60_000), replace=False)
                full = embr.embed([texts[i] for i in idx], EMB_BATCH)
                mu, W, expl = er_embed.fit_pca(full, EMB_DIM)
                fid = er_embed.pca_fidelity(full, mu, W, rng)
                np.savez(pca_p, mu=mu, W=W, explained=expl, **fid)
                del full, texts
            mu, W = (np.load(pca_p)['mu'], np.load(pca_p)['W']) if EMB_DIM else (None, None)
            for split in ('train', 'test'):
                t0 = time.time()
                texts = field_texts(rec[split], field)
                er_embed.embed_to_memmap(embr, texts, ed / f'{split}_{field}.npy', mu, W, EMB_BATCH)
                dt = max(time.time() - t0, 1e-9)
                EMB_RATES[f'{split}/{field}'] = len(texts) / dt
                print(f'  {split}/{field}: {len(texts):,} records in {dt:,.0f}s ({len(texts) / dt:,.0f} rec/s, incl. resumed chunks)')
                del texts; gc.collect()
        embr.close(); del embr; gc.collect()
        mark_done(ed)
    E = {s: {f: np.load(ed / f'{s}_{f}.npy', mmap_mode='r') for f in EMB_FIELDS} for s in ('train', 'test')}
    for s in E:
        for f in E[s]:
            assert E[s][f].shape[0] == len(rec[s]), (s, f, E[s][f].shape, len(rec[s]))
    PCA_INFO = {}
    if EMB_DIM:
        for f in EMB_FIELDS:
            z = np.load(ed / f'pca_{f}.npz')
            PCA_INFO[f] = {k: round(float(z[k]), 4) for k in ('explained', 'rand_corr', 'nn_corr', 'nn_top1_kept')}
        print('PCA fidelity (explained variance; cosine correlation full vs projected; nearest neighbour kept):', PCA_INFO)
    # sanity: true pairs must be much closer than random pairs
    k = min(len(POS_KEYS), 20000)
    pk = rng.choice(POS_KEYS, k, replace=False)
    qq, tt = pk // NR_TR, pk % NR_TR
    sims = {}
    for f in EMB_FIELDS:
        cp = er_block.gathered_cos(E['train'][f], qq, tt)
        cr = er_block.gathered_cos(E['train'][f], qq, rng.integers(0, NR_TR, k))
        sims[f] = (round(float(np.median(cp)), 3), round(float(np.median(cr)), 3))
        print(f'  {f:5s}: median cosine true pairs {sims[f][0]:.3f} vs random pairs {sims[f][1]:.3f}')
    ok = check(all(a > b + 0.15 for a, b in sims.values()), 'true pairs are far closer than random pairs in every field')
    dim = E['test'][EMB_FIELDS[0]].shape[1]
    rates = {k_: round(v) for k_, v in EMB_RATES.items()}
    paste(f'cell6 dim={dim} pca={PCA_INFO} true_vs_random={sims} rates={rates} pass={ok}')

── 6 embeddings ── start (RSS 6.04 GB)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

    train_name.npy: 4,596/4,596  453 rec/s  ETA 0.0 min
  train/name: 4,596 records in 10s (452 rec/s, incl. resumed chunks)
    test_name.npy: 49,163/49,163  456 rec/s  ETA 0.0 min
  test/name: 49,163 records in 108s (456 rec/s, incl. resumed chunks)
    train_addr.npy: 4,596/4,596  284 rec/s  ETA 0.0 min
  train/addr: 4,596 records in 16s (283 rec/s, incl. resumed chunks)
    test_addr.npy: 49,163/49,163  271 rec/s  ETA 0.0 min
  test/addr: 49,163 records in 181s (271 rec/s, incl. resumed chunks)
    train_combo.npy: 4,596/4,596  201 rec/s  ETA 0.0 min
  train/combo: 4,596 records in 23s (200 rec/s, incl. resumed chunks)
    test_combo.npy: 49,163/49,163  201 rec/s  ETA 0.0 min
  test/combo: 49,163 records in 245s (200 rec/s, incl. resumed chunks)
PCA fidelity (explained variance; cosine correlation full vs projected; nearest neighbour kept): {'name': {'explained': 0.8869, 'rand_corr': 0.7067, 'nn_corr': 0.9662, 'nn_top1_kept': 0.7663}, 'addr': {'explained': 0.9176, 'rand_corr': 0.89

## Cell 7 — Blocking

For each S1, candidates are the union over S2 ∪ S3 of:
* **(a)** char_wb 2–4 TF-IDF on core names, top `K_TFIDF`, using chunked sparse matmul and per-row top-k;
* **(a2)** word TF-IDF on name+address, top `K_COMBO` (an addition: a strong retriever in the earlier dev run);
* **(b)** dense top `K_DENSE` in MLX, using float16 `chunk @ targets.T`, `mx.argpartition` and `mx.eval` per chunk;
* **(c)** rare-token and postal keys, top `K_RARE`.

When `SAME_COUNTRY`, targets are restricted to the same country string. On train it reports pair recall, per-S1 full recall, mean candidates and reduction ratio, per retriever and for the union, per country.

In [20]:
def groups_of(r):
    src = r['src'].to_numpy()
    ctry = r['country'].astype(str).to_numpy(dtype=object)
    keys = sorted(set(ctry)) if SAME_COUNTRY else ['ALL']   # open set: whatever country strings exist
    out = {}
    for g in keys:
        m = np.ones(len(r), bool) if g == 'ALL' else (ctry == g)
        out[g] = (np.flatnonzero(m & (src == 1)), np.flatnonzero(m & (src != 1)))
    return out

GROUPS = {s: groups_of(rec[s]) for s in rec}
E_RET = {s: (E[s]['combo'] if 'combo' in E[s] else er_block.ConcatEmb(E[s]['name'], E[s]['addr'])) for s in rec}
BLOCK_P = {'emb': EMB_P, 'cfg': CFG_B, 'SAME_COUNTRY': SAME_COUNTRY, 'v': 2}
with section('7 blocking'):
    bd, fresh = stage_dir(ART, 'block', BLOCK_P, FORCE)
    for split in ('train', 'test'):
        for gi, (g, (qi, ti)) in enumerate(GROUPS[split].items()):
            p = bd / f'{split}_{gi:02d}.parquet'
            if p.exists():   # same params (stage_dir wipes on change) -> resume per group
                continue
            print(f'  {split} / {g}: {len(qi):,} S1 x {len(ti):,} S2uS3')
            u = er_block.block_group(rec[split], qi, ti, E_RET[split], CFG_B, N_JOBS)
            u.to_parquet(p, index=False)
            del u; gc.collect()
    mark_done(bd)

    ptr = pd.concat([pd.read_parquet(bd / f'train_{gi:02d}.parquet') for gi in range(len(GROUPS['train']))], ignore_index=True)
    rep, rep_k = er_block.recall_report(ptr, rec['train'], POS_KEYS, N_TRUE, GROUPS['train'])
    print('train blocking per country group (pair recall, per-S1 full recall, mean candidates, reduction ratio):')
    print(rep.round(4).to_string(index=False))
    key = ptr['q'].to_numpy(np.int64) * NR_TR + ptr['t'].to_numpy(np.int64)
    is_pos = np.isin(key, POS_KEYS)
    BLOCK_RECALL = float(is_pos.sum() / max(len(POS_KEYS), 1))
    nonsing = np.flatnonzero((rec['train']['src'].to_numpy() == 1) & (N_TRUE > 0))
    found = np.bincount(ptr['q'].to_numpy()[is_pos], minlength=NR_TR)
    BLOCK_FULL_S1 = float((found[nonsing] == N_TRUE[nonsing]).mean())
    n_s1_tr = int((rec['train']['src'].to_numpy() == 1).sum())
    BLOCK_CANDS = len(ptr) / max(n_s1_tr, 1)
    rk = {}
    for rname in er_block.RET:
        r_ = ptr[f'r_{rname}'].to_numpy()
        kmax = int(r_.max()) if len(r_) else 0
        rk[rname] = {kk: round(float(((r_ > 0) & (r_ <= kk) & is_pos).sum() / max(len(POS_KEYS), 1)), 4)
                     for kk in sorted({5, 10, 20, 30, 50, kmax}) if 0 < kk <= kmax}
    print('overall pair recall at k per retriever:', rk)
    ok = check(BLOCK_RECALL >= 0.98, f'union pair recall {BLOCK_RECALL:.4f} (target >= 0.98); S1 full-recall {BLOCK_FULL_S1:.4f}; {BLOCK_CANDS:.1f} candidates per S1')
    if not ok:
        for rname, curve in rk.items():
            ks_ = sorted(curve)
            if len(ks_) >= 2 and curve[ks_[-1]] - curve[ks_[-2]] > 0.003:
                print(f'  suggestion: {rname} still gains {curve[ks_[-1]] - curve[ks_[-2]]:.4f} between k={ks_[-2]} and k={ks_[-1]} -> double its k')
        print('  also consider raising TFIDF_MAX_DF_MIN, PRUNE_CHAR / PRUNE_WORD, RARE_MAX_DF')
    test_stats = {}
    for gi, (g, (qi, ti)) in enumerate(GROUPS['test'].items()):
        qcol = pd.read_parquet(bd / f'test_{gi:02d}.parquet', columns=['q'])['q'].to_numpy()
        test_stats[g] = {'S1': len(qi), 'S2uS3': len(ti), 'cands_per_S1': round(len(qcol) / max(len(qi), 1), 1),
                         'S1_without_cands': round(1 - len(np.unique(qcol)) / max(len(qi), 1), 4)}
    print('test candidates:', test_stats)
    by_grp = {r_['group'] + '/' + r_['retriever']: round(r_['pair_recall'], 4) for _, r_ in rep.iterrows()}
    paste(f'cell7 union_recall={BLOCK_RECALL:.4f} s1_full={BLOCK_FULL_S1:.4f} cands/S1_train={BLOCK_CANDS:.1f} recall={by_grp} recall_at_k={rk} test={test_stats} pass={ok}')
    del ptr, key, is_pos; gc.collect()

── 7 blocking ── start (RSS 0.42 GB)
  train / India: 120 S1 x 1,729 S2uS3
      tfidf  0s  vocab=5,065
      combo  0s  vocab=5,564
      rare   0s  rare vocab=5,522
      dense  0s
      union  0s  pairs=2,449  (20.4/S1)
  train / US: 180 S1 x 2,567 S2uS3
      tfidf  0s  vocab=7,731
      combo  0s  vocab=7,073
      rare   0s  rare vocab=7,364
      dense  0s
      union  0s  pairs=3,375  (18.8/S1)
  test / France: 30 S1 x 7,645 S2uS3
      tfidf  0s  vocab=7,991
      combo  0s  vocab=9,022
      rare   0s  rare vocab=8,870
      dense  0s
      union  0s  pairs=782  (26.1/S1)
  test / India: 94 S1 x 24,990 S2uS3
      tfidf  0s  vocab=21,121
      combo  1s  vocab=39,707
      rare   1s  rare vocab=40,225
      dense  1s
      union  1s  pairs=2,612  (27.8/S1)
  test / US: 77 S1 x 16,327 S2uS3
      tfidf  0s  vocab=17,182
      combo  0s  vocab=26,638
      rare   1s  rare vocab=28,842
      dense  1s
      union  1s  pairs=2,068  (26.9/S1)
train blocking per country group (pair

## Cell 8 — Pair features (CPU, parallel; dense cosines gathered in MLX)

* **Names**, both raw-normalised and suffix-stripped: ratio, partial_ratio, token_sort, token_set, Jaro-Winkler and token Jaccard. Also char-TF-IDF cosine, dense name cosine, legal suffix match/conflict/missing, acronym, length ratio and name frequency.
* **Addresses:** the same string similarities, dense cosine, Jaccard, postal and street-number equal/conflict/missing, digit-set Jaccard, and landmark flags.
* **Context:** source, rank and reverse rank (dense / TF-IDF / combo), mutual-best, gap to the next best, candidate counts, and max similarity to the S1's top candidates from the other source.

String scores use rapidfuzz `cpdist` (multi-threaded C); Jaccards use sparse row-dots. **Country is not a feature.**

In [21]:
import pyarrow.parquet as pq_
FEAT_P = {'block': BLOCK_P, 'FEAT_CHUNK': FEAT_CHUNK, 'v': 2}
with section('8 features'):
    fd, fresh = stage_dir(ART, 'feat', FEAT_P, FORCE)
    FEAT_FILES = {}
    for split in ('train', 'test'):
        (fd / split).mkdir(exist_ok=True)
        side, files = None, []
        for gi, g in enumerate(GROUPS[split]):
            tag = f'g{gi:02d}'
            flag = fd / split / f'{tag}.done'
            if not flag.exists():   # per-group resume after an interruption
                if side is None:
                    side = er_feats.record_side_features(rec[split])
                pairs = pd.read_parquet(bd / f'{split}_{gi:02d}.parquet')
                t0 = time.time()
                er_feats.featurize_group(rec[split], side, pairs, E_RET[split], {f: E[split][f] for f in EMB_FIELDS},
                                         POS_KEYS if split == 'train' else None, fd / split, tag, N_JOBS, FEAT_CHUNK)
                dt = max(time.time() - t0, 1e-9)
                print(f'  {split}/{g}: {len(pairs):,} pairs in {dt:,.0f}s ({len(pairs) / dt:,.0f} pairs/s)')
                flag.write_text('ok')
                del pairs; gc.collect()
            files += sorted((fd / split).glob(f'{tag}_*.parquet'))
        FEAT_FILES[split] = files
        del side; gc.collect()
    mark_done(fd)
    peek = pd.read_parquet(FEAT_FILES['train'][0])
    FEATS = er_feats.feature_columns(peek)
    assert not any('country' in c for c in FEATS), 'country must never be a feature'
    n_rows = {s: sum(pq_.ParquetFile(f).metadata.num_rows for f in FEAT_FILES[s]) for s in FEAT_FILES}
    print(f'{len(FEATS)} features: {FEATS}')
    print('highest NaN rates:', peek[FEATS].isna().mean().sort_values(ascending=False).head(8).round(3).to_dict())
    show = [c for c in ('c_tset', 'n_jw', 'a_tset', 'cos_name', 'cos_addr', 's_dense', 'rank_s_dense', 'postal_eq', 'corrob_max') if c in FEATS]
    print('feature means by label (first train part):')
    print(peek.groupby('y')[show].mean().round(3).T.to_string())
    pos_rate = float(peek['y'].mean())
    ok = check(len(FEATS) >= 40 and n_rows['train'] > 0 and n_rows['test'] > 0,
               f"{len(FEATS)} features; train pairs {n_rows['train']:,}, test pairs {n_rows['test']:,}")
    paste(f"cell8 n_feats={len(FEATS)} train_pairs={n_rows['train']} test_pairs={n_rows['test']} pos_rate_part0={pos_rate:.4f} pass={ok}")
    del peek

── 8 features ── start (RSS 0.57 GB)
      g00 part 0: 2,449 pairs
  train/India: 2,449 pairs in 0s (36,438 pairs/s)
      g01 part 0: 3,375 pairs
  train/US: 3,375 pairs in 0s (49,487 pairs/s)
      g00 part 0: 782 pairs
  test/France: 782 pairs in 0s (5,817 pairs/s)
      g01 part 0: 2,612 pairs
  test/India: 2,612 pairs in 0s (24,052 pairs/s)
      g02 part 0: 2,068 pairs
  test/US: 2,068 pairs in 0s (34,561 pairs/s)
65 features: ['is_s3', 's_dense', 's_tfidf', 's_combo', 's_rare', 'rank_s_dense', 'revrank_s_dense', 'gap_s_dense', 'revgap_s_dense', 'rank_s_tfidf', 'revrank_s_tfidf', 'gap_s_tfidf', 'revgap_s_tfidf', 'rank_s_combo', 'revrank_s_combo', 'gap_s_combo', 'revgap_s_combo', 'mutual_best', 'mutual_best_tfidf', 'n_cands', 'n_rev', 'hit_tfidf', 'hit_combo', 'hit_dense', 'hit_rare', 'n_hits', 'corrob_max', 'n_ratio', 'n_partial', 'n_tsort', 'n_tset', 'n_jw', 'c_ratio', 'c_partial', 'c_tsort', 'c_tset', 'c_jw', 'a_ratio', 'a_partial', 'a_tsort', 'a_tset', 'a_jw', 'n_jacc', 'c_jac

## Cell 9 — Cross-validation splits

The main scheme is `GroupKFold(N_FOLDS)` grouped by S1. As a France proxy, **leave-one-country-out** trains on the other countries and evaluates on the held-out one; it runs for both models in cells 10 and 11.

In [22]:
from sklearn.model_selection import GroupKFold
with section('9 CV splits'):
    TR = pd.concat([pd.read_parquet(f, columns=['q', 't', 'y'] + FEATS) for f in FEAT_FILES['train']], ignore_index=True)
    q_tr, t_tr = TR['q'].to_numpy(np.int64), TR['t'].to_numpy(np.int64)
    y_tr = TR['y'].to_numpy(np.int8)
    X_tr = TR[FEATS].to_numpy(np.float32)
    del TR; gc.collect()
    folds = np.full(len(y_tr), -1, np.int8)
    for f, (_, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(X_tr[:, :1], y_tr, groups=q_tr)):
        folds[va] = f
    ctry_tr = rec['train']['country'].astype(str).to_numpy(dtype=object)
    PAIR_CTRY = ctry_tr[q_tr]
    S1_ROWS = np.flatnonzero(rec['train']['src'].to_numpy() == 1)   # every sampled train S1 (no-candidate S1s predict empty)
    S1_CTRY = ctry_tr[S1_ROWS]
    COUNTRIES = sorted(set(S1_CTRY))
    LOCO = [(c, PAIR_CTRY != c, PAIR_CTRY == c) for c in COUNTRIES] if len(COUNTRIES) > 1 else []
    LOCO_RES, CV_SCORES = {}, {}

    def loco_eval(model_name, c, p, vam):
        """Held-out country scores from raw probabilities (nothing fitted on the held-out country)."""
        qv, tv, yv = q_tr[vam], t_tr[vam], y_tr[vam]
        rows_c = S1_ROWS[S1_CTRY == c]
        auc, pr = er_models.auc_pr(yv, p)
        pa = er_decide.assign_one_to_one(qv, tv, p) if ONE_TO_ONE else p
        f_thr = er_decide.f05_vec(qv, pa >= 0.5, yv, N_TRUE, rows_c).mean()
        sel = er_decide.expected_f_select(qv, pa, seed=SEED, draws=MC_DRAWS) & (pa > 0)
        f_ef = er_decide.f05_vec(qv, sel, yv, N_TRUE, rows_c).mean()
        return {'model': model_name, 'train_on': '+'.join(x for x in COUNTRIES if x != c), 'eval_on': c,
                'auc': round(auc, 5), 'pr_auc': round(pr, 5), 'macroF05_thr0.5': round(f_thr, 5),
                'macroF05_expectedF': round(f_ef, 5)}

    per_q = int(pd.Series(folds).groupby(q_tr).nunique().max()) if len(folds) else 0
    fold_tab = pd.DataFrame({'fold': folds, 'y': y_tr, 'q': q_tr}).groupby('fold').agg(
        pairs=('y', 'size'), pos_rate=('y', 'mean'), n_s1=('q', 'nunique'))
    print(fold_tab.round(4).to_string())
    print(f'X_tr {X_tr.shape} float32 = {X_tr.nbytes / 2**30:.2f} GB; leave-country-out splits: {[c for c, _, _ in LOCO]}')
    ok = check((folds >= 0).all() and per_q == 1, f'{N_FOLDS} folds grouped by S1 (max folds per S1 = {per_q})')
    paste(f'cell9 pairs={len(y_tr)} pos_rate={y_tr.mean():.4f} folds={N_FOLDS} loco={[c for c, _, _ in LOCO]} X_GB={X_tr.nbytes / 2**30:.2f} pass={ok}')

── 9 CV splits ── start (RSS 0.66 GB)
      pairs  pos_rate  n_s1
fold                       
0      1942    0.1807   100
1      1941    0.1736   100
2      1941    0.1989   100
X_tr (5824, 65) float32 = 0.00 GB; leave-country-out splits: ['India', 'US']
✅ PASS: 3 folds grouped by S1 (max folds per S1 = 1)
>>> PASTE TO CLAUDE: cell9 pairs=5824 pos_rate=0.1844 folds=3 loco=['India', 'US'] X_GB=0.00 pass=True
── 9 CV splits ── 0.1s | RSS 0.67 GB | peak RSS 6.91 GB


## Cell 10 — LightGBM on scalar features (CPU)

Runs with `num_threads=N_JOBS` (performance cores). It produces OOF predictions via GroupKFold, test predictions averaged over the fold models, AUC / PR-AUC overall and per country, and the top feature importances. Leave-country-out uses the median best iteration from CV.

In [23]:
# LightGBM runs in a clean subprocess (er_lgb_worker.py): on macOS a second OpenMP runtime in this kernel
# (torch's or scikit-learn's libomp) segfaults LightGBM's thread pool. The worker loads only numpy/pandas/lightgbm.
LGB_P = {'feat': FEAT_P, 'N_FOLDS': N_FOLDS, 'LGB_ROUNDS': LGB_ROUNDS, 'LGB_ES': LGB_ES, 'SEED': SEED,
         'ONE_TO_ONE': ONE_TO_ONE, 'MC_DRAWS': MC_DRAWS, 'SMOKE': SMOKE, 'v': 2}
with section('10 LightGBM'):
    ld, fresh = stage_dir(ART, 'lgb', LGB_P, FORCE)
    params = er_models.lgb_params(SEED, N_JOBS, SMOKE)
    loco_countries = [c for c, _, _ in LOCO]
    if not fresh:
        np.save(ld / 'X.npy', X_tr)
        np.save(ld / 'y.npy', y_tr)
        np.save(ld / 'folds.npy', folds)
        np.save(ld / 'pair_ctry.npy', np.asarray(PAIR_CTRY, dtype=str))
        job = {'params': params, 'feats': FEATS, 'rounds': LGB_ROUNDS, 'es_rounds': LGB_ES,
               'test_files': [str(p) for p in FEAT_FILES['test']], 'loco_countries': loco_countries}
        (ld / 'job.json').write_text(json.dumps(job))
        proc = subprocess.Popen([sys.executable, '-u', str(ROOT / 'er_lgb_worker.py'), str(ld)], cwd=ROOT,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='', flush=True)
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f'LightGBM worker exited with code {rc}' + (' (segfault in the worker)' if rc == -11 else '')
                               + '; the kernel is still alive, paste the output above')
        (ld / 'X.npy').unlink()   # large copy no longer needed
        mark_done(ld)
    oof_lgb = np.load(ld / 'oof.npy')
    info_lgb = json.loads((ld / 'info.json').read_text())
    imp = pd.Series(np.load(ld / 'importance.npy'), index=FEATS).sort_values(ascending=False)
    LOCO_RES['lgb'] = pd.DataFrame([loco_eval('lgb', c, np.load(ld / f'loco_{i}.npy'), vam)
                                    for i, (c, _, vam) in enumerate(LOCO)])
    auc, pr = er_models.auc_pr(y_tr, oof_lgb)
    per_c = {c: tuple(round(v, 5) for v in er_models.auc_pr(y_tr[PAIR_CTRY == c], oof_lgb[PAIR_CTRY == c])) for c in COUNTRIES}
    th_l, f_lgb = er_decide.best_threshold(q_tr, oof_lgb, y_tr, N_TRUE, S1_ROWS)
    CV_SCORES['lgb'] = f_lgb
    print(f"best iterations per fold: {info_lgb['best_iters']}")
    print(f'LightGBM OOF auc={auc:.5f} pr_auc={pr:.5f}; per country (auc, pr_auc): {per_c}')
    print(f'macro-F0.5 at best global threshold {th_l}: {f_lgb:.5f}')
    print('top 25 features (mean gain):')
    print(imp.head(25).round(1).to_string())
    print(LOCO_RES['lgb'].to_string(index=False))
    ok = check(auc > 0.95, f'LightGBM OOF AUC {auc:.5f}')
    paste(f"cell10 lgb oof auc={auc:.5f} pr_auc={pr:.5f} per_country={per_c} macroF05={f_lgb:.5f}@thr{th_l} loco={LOCO_RES['lgb'].to_dict('records')} top8={list(imp.index[:8])} pass={ok}")

── 10 LightGBM ── start (RSS 0.67 GB)
worker: X (5824, 65), lightgbm 4.7.0, threads 8
    fold 0: best_iter=150 (0s)
    fold 1: best_iter=106 (0s)
    fold 2: best_iter=84 (0s)
    test: 5,462 pairs scored
    LOCO eval on India: 106 rounds (0s)
    LOCO eval on US: 106 rounds (0s)
worker: done
best iterations per fold: [150, 106, 84]
LightGBM OOF auc=0.99990 pr_auc=0.99958; per country (auc, pr_auc): {'India': (0.99986, 0.99937), 'US': (0.99994, 0.99975)}
macro-F0.5 at best global threshold 0.49: 0.99551
top 25 features (mean gain):
s_combo           11496.0
s_dense            1401.6
cos_combo          1350.6
n_hits              307.1
a_tset              234.4
a_partial            93.0
d_jacc               81.0
revgap_s_combo       80.5
revgap_s_dense       75.4
cos_name             70.7
c_jw                 58.0
cos_addr             35.2
c_tsort              27.7
corrob_max           27.0
a_ratio              25.1
n_jw                 25.1
c_ratio              24.9
revgap_s_tfidf   

## Cell 11 — Siamese tower MLP in MLX

**Inputs** are index-based: (s1_idx, cand_idx) plus standardised scalar features. The scaler is fit on the training fold only. Frozen name and address embeddings sit in unified memory as float16 `mx.array`s; they are looked up with `mx.take` and cast to fp32 inside `forward`.

**Model:**
* Name tower: `[|a−b|, a*b, cos]` → Linear → GELU → LayerNorm → Dropout, 256 out.
* Address tower: the same, on the address embeddings.
* Scalar tower: Linear → 128.
* Head: concat → 512 → 128 → 1 logit.

**Training:** BCE with logits, with pos_weight 1 vs √(neg/pos) compared on OOF PR-AUC. AdamW (lr 1e-3, wd 1e-4) with cosine decay, batch 1024, fp32. The step is built with `nn.value_and_grad` inside `mx.compile`. Early stopping uses fold-validation PR-AUC.

`BACKEND='torch_mps'` runs the same model in PyTorch on MPS (fp32).

In [24]:
MLP_P = {'feat': FEAT_P, 'N_FOLDS': N_FOLDS, 'BACKEND': BACKEND, 'MLP_EPOCHS': MLP_EPOCHS, 'MLP_BS': MLP_BS,
         'MLP_COMPARE_PW': MLP_COMPARE_PW, 'SEED': SEED, 'ONE_TO_ONE': ONE_TO_ONE, 'MC_DRAWS': MC_DRAWS, 'v': 1}
with section('11 MLP'):
    md, fresh = stage_dir(ART, 'mlp', MLP_P, FORCE)
    if not fresh:
        if BACKEND == 'mlx':
            # frozen embedding tables live in unified memory as float16; the model looks rows up with mx.take
            EN_tr, EA_tr = mx.array(np.asarray(E['train']['name'])), mx.array(np.asarray(E['train']['addr']))
            train_fold = er_models.mlx_train_fold
            predict_tr = lambda mdl, q, t, Z: er_models.mlx_predict(mdl, EN_tr, EA_tr, q, t, Z)
            predict_te = lambda mdl, q, t, Z: er_models.mlx_predict(mdl, E['test']['name'], E['test']['addr'], q, t, Z)
        elif BACKEND == 'torch_mps':
            EN_tr, EA_tr = np.asarray(E['train']['name']), np.asarray(E['train']['addr'])
            train_fold = er_models.torch_train_fold
            predict_tr = lambda mdl, q, t, Z: mdl.predict_np(q, t, Z)
            predict_te = lambda mdl, q, t, Z: er_models.torch_predict_test(mdl, E['test']['name'], E['test']['addr'], q, t, Z)
        else:
            raise ValueError(f'BACKEND must be mlx or torch_mps, got {BACKEND}')
        npos = int(y_tr.sum())
        pw_auto = round(float(np.sqrt((len(y_tr) - npos) / max(npos, 1))), 3)
        pw_list = [1.0, pw_auto] if MLP_COMPARE_PW else [1.0]
        runs = {}
        for pw in pw_list:
            oof, fm, hists = np.zeros(len(y_tr), np.float32), [], []
            for f in range(N_FOLDS):
                tr, va = folds != f, folds == f
                sc = er_models.Scaler().fit(X_tr[tr])
                Z_va = sc.transform(X_tr[va])
                mdl, hist = train_fold(EN_tr, EA_tr, q_tr[tr], t_tr[tr], sc.transform(X_tr[tr]), y_tr[tr],
                                       q_tr[va], t_tr[va], Z_va, y_tr[va], epochs=MLP_EPOCHS, bs=MLP_BS,
                                       pos_weight=pw, seed=SEED + f)
                oof[va] = predict_tr(mdl, q_tr[va], t_tr[va], Z_va)
                fm.append((mdl, sc))
                hists.append(hist)
                spe = np.mean([h['sec'] for h in hist])
                print(f'    pos_weight={pw} fold {f}: {len(hist)} epochs, {spe:.1f} s/epoch')
                del Z_va; gc.collect()
            auc, pr = er_models.auc_pr(y_tr, oof)
            runs[pw] = {'oof': oof, 'models': fm, 'auc': auc, 'pr': pr, 'hists': hists}
            print(f'  pos_weight={pw}: OOF auc={auc:.5f} pr_auc={pr:.5f}')
        best_pw = max(runs, key=lambda k_: runs[k_]['pr'])
        oof_mlp, fold_models = runs[best_pw]['oof'], runs[best_pw]['models']
        np.save(md / 'oof.npy', oof_mlp)
        p_te = []
        for fpath in FEAT_FILES['test']:
            d = pd.read_parquet(fpath, columns=['q', 't'] + FEATS)
            qx, tx, Xt = d['q'].to_numpy(np.int64), d['t'].to_numpy(np.int64), d[FEATS].to_numpy(np.float32)
            del d
            p_te.append(np.mean([predict_te(mdl, qx, tx, sc.transform(Xt)) for mdl, sc in fold_models], axis=0).astype(np.float32))
        np.save(md / 'test.npy', np.concatenate(p_te) if p_te else np.zeros(0, np.float32))
        # leave-one-country-out with a fixed epoch count (median best epoch of the CV folds)
        best_eps = [int(np.argmax([h['pr_auc'] for h in hist])) + 1 for hist in runs[best_pw]['hists']]
        n_ep = max(1, int(np.median(best_eps)))
        loco = []
        for c, trm, vam in LOCO:
            sc = er_models.Scaler().fit(X_tr[trm])
            tri = np.flatnonzero(trm)
            mon = np.random.default_rng(SEED).choice(tri, max(1, len(tri) // 20), replace=False)  # in-sample monitor only
            mdl, _ = train_fold(EN_tr, EA_tr, q_tr[trm], t_tr[trm], sc.transform(X_tr[trm]), y_tr[trm],
                                q_tr[mon], t_tr[mon], sc.transform(X_tr[mon]), y_tr[mon], epochs=n_ep, bs=MLP_BS,
                                pos_weight=best_pw, patience=n_ep + 1, seed=SEED)
            loco.append(loco_eval('mlp', c, predict_tr(mdl, q_tr[vam], t_tr[vam], sc.transform(X_tr[vam])), vam))
        pd.DataFrame(loco).to_csv(md / 'loco.csv', index=False)
        info = {'best_pos_weight': best_pw, 'loco_epochs': n_ep,
                'runs': {str(k_): {'auc': v['auc'], 'pr_auc': v['pr'], 'epochs_run': [len(h) for h in v['hists']],
                                   'sec_per_epoch': float(np.mean([x['sec'] for h in v['hists'] for x in h]))} for k_, v in runs.items()}}
        json.dump(info, open(md / 'info.json', 'w'), indent=1)
        del runs, fold_models, EN_tr, EA_tr
        gc.collect(); mx.clear_cache()
        mark_done(md)
    oof_mlp = np.load(md / 'oof.npy')
    info = json.load(open(md / 'info.json'))
    LOCO_RES['mlp'] = pd.read_csv(md / 'loco.csv') if LOCO else pd.DataFrame()
    auc, pr = er_models.auc_pr(y_tr, oof_mlp)
    th_m, f_mlp = er_decide.best_threshold(q_tr, oof_mlp, y_tr, N_TRUE, S1_ROWS)
    CV_SCORES['mlp'] = f_mlp
    print(json.dumps(info, indent=1))
    print(f'MLP OOF auc={auc:.5f} pr_auc={pr:.5f} macro-F0.5 at best threshold {th_m}: {f_mlp:.5f}')
    print(LOCO_RES['mlp'].to_string(index=False))
    ok = check(auc > 0.95, f'MLP ({BACKEND}) OOF AUC {auc:.5f}')
    spe = {k_: round(v['sec_per_epoch'], 1) for k_, v in info['runs'].items()}
    pw_cmp = {k_: round(v['pr_auc'], 5) for k_, v in info['runs'].items()}
    paste(f"cell11 backend={BACKEND} oof auc={auc:.5f} pr_auc={pr:.5f} macroF05={f_mlp:.5f} pos_weight_prauc={pw_cmp} best_pw={info['best_pos_weight']} sec_per_epoch={spe} loco={LOCO_RES['mlp'].to_dict('records')} pass={ok}")

── 11 MLP ── start (RSS 0.67 GB)
      epoch 0: val auc=0.99996 pr_auc=0.99985  0.7s/epoch
      epoch 1: val auc=0.99998 pr_auc=0.99990  0.0s/epoch
      epoch 2: val auc=0.99999 pr_auc=0.99994  0.0s/epoch
      epoch 3: val auc=0.99999 pr_auc=0.99995  0.1s/epoch
    pos_weight=1.0 fold 0: 4 epochs, 0.2 s/epoch
      epoch 0: val auc=0.99989 pr_auc=0.99953  0.1s/epoch
      epoch 1: val auc=0.99991 pr_auc=0.99959  0.0s/epoch
      epoch 2: val auc=0.99992 pr_auc=0.99965  0.1s/epoch
      epoch 3: val auc=0.99992 pr_auc=0.99964  0.1s/epoch
    pos_weight=1.0 fold 1: 4 epochs, 0.1 s/epoch
      epoch 0: val auc=0.99980 pr_auc=0.99926  0.1s/epoch
      epoch 1: val auc=0.99985 pr_auc=0.99944  0.0s/epoch
      epoch 2: val auc=0.99986 pr_auc=0.99948  0.1s/epoch
      epoch 3: val auc=0.99986 pr_auc=0.99947  0.1s/epoch
    pos_weight=1.0 fold 2: 4 epochs, 0.1 s/epoch
  pos_weight=1.0: OOF auc=0.99988 pr_auc=0.99952
      epoch 0: val auc=0.99992 pr_auc=0.99967  0.1s/epoch
      epoch 1: va

## Cell 12 — Blend, calibrate and decide

1. **Blend:** weight w for LightGBM vs the MLP, chosen on OOF macro-F0.5 (grid 0…1).
2. **Calibrate:** isotonic regression on the blended OOF probabilities. It is cross-fitted by fold for evaluation, and the final fit uses all OOF and is applied to test.
3. **Assignment** (if `ONE_TO_ONE`): each S2/S3 record keeps only its highest-probability S1.
4. **Expected F0.5:** per S1, choose k = 0…min(K, 10) maximising the Monte-Carlo expected F0.5 of its top-k candidates. Candidates are treated as independent Bernoulli draws, and k = 0 scores 1 only when no candidate is a true match.
5. **Compare** {threshold, threshold+assign, expectedF, expectedF+assign} on OOF, overall and per country. Singleton accuracy and non-singleton F0.5 are reported separately. **Nothing is tuned on test.**

In [25]:
with section('12 blend/calibrate/decide'):
    # 1. blend weight on OOF macro-F0.5 (best global threshold per weight; isotonic is monotone so this is equivalent)
    w_lgb, blend_tab = er_decide.choose_blend(q_tr, oof_lgb, oof_mlp, y_tr, N_TRUE, S1_ROWS)
    print('blend grid (OOF):')
    print(blend_tab.round(5).to_string(index=False))
    CV_SCORES['blend'] = float(blend_tab['macro_f05'].max())
    p_oof = (w_lgb * oof_lgb + (1 - w_lgb) * oof_mlp).astype(np.float32)
    # 2. isotonic calibration: cross-fitted on OOF for honest evaluation; the final map is fit on all OOF, applied to test
    p_cal = er_decide.isotonic_crossfit(p_oof, y_tr, folds)
    ISO = er_decide.isotonic_fit(p_oof, y_tr)
    # 3-5. decision variants on OOF (overall + per country)
    var_tab, _, thr = er_decide.evaluate_variants(q_tr, t_tr, p_cal, y_tr, N_TRUE, S1_ROWS, S1_CTRY, SEED, MC_DRAWS)
    print(var_tab.round(5).to_string(index=False))
    allowed = var_tab[var_tab['group'] == 'all']
    if not ONE_TO_ONE:
        allowed = allowed[~allowed['variant'].str.contains('assign')]
    best = allowed.loc[allowed['macro_F0.5'].idxmax()]
    best_variant = str(best['variant'])
    best_thr = thr.get(best_variant)
    DECISION = {'w_lgb': float(w_lgb), 'variant': best_variant, 'thr': None if best_thr is None else float(best_thr),
                'one_to_one': bool(ONE_TO_ONE), 'oof_macro_f05': float(best['macro_F0.5'])}
    json.dump(DECISION, open(ART / 'decision.json', 'w'), indent=1)
    print('DECISION', DECISION)
    print(f'upper bound from blocking: union pair recall {BLOCK_RECALL:.4f}, S1 full recall {BLOCK_FULL_S1:.4f}')
    bc = var_tab[var_tab['variant'] == best_variant].set_index('group')
    per_c = {g: (round(r_['macro_F0.5'], 4), round(r_['singleton_acc'], 4), round(r_['nonsingleton_F0.5'], 4)) for g, r_ in bc.iterrows()}
    ok = check(DECISION['oof_macro_f05'] > 0.5, f"chosen rule {best_variant}: OOF macro-F0.5 {DECISION['oof_macro_f05']:.5f}")
    others = allowed.set_index('variant')['macro_F0.5'].round(5).to_dict()
    paste(f"cell12 w_lgb={w_lgb} rule={best_variant} thr={DECISION['thr']} oofF05={DECISION['oof_macro_f05']:.5f} variants={others} per_group(F,singleton_acc,nonsing_F)={per_c} pass={ok}")

── 12 blend/calibrate/decide ── start (RSS 0.65 GB)
blend grid (OOF):
 w_lgb  thr  macro_f05
   0.0 0.64    0.99635
   0.1 0.68    0.99694
   0.2 0.62    0.99694
   0.3 0.71    0.99697
   0.4 0.64    0.99697
   0.5 0.69    0.99676
   0.6 0.75    0.99676
   0.7 0.81    0.99660
   0.8 0.40    0.99571
   0.9 0.45    0.99551
   1.0 0.49    0.99551
         variant group  n_s1  macro_F0.5  singleton_acc  nonsingleton_F0.5  threshold
       threshold   all   300     0.99678            1.0            0.99661       0.75
       threshold India   120     0.99548            1.0            0.99524       0.75
       threshold    US   180     0.99765            1.0            0.99753       0.75
threshold+assign   all   300     0.99678            1.0            0.99661       0.75
threshold+assign India   120     0.99548            1.0            0.99524       0.75
threshold+assign    US   180     0.99765            1.0            0.99753       0.75
       expectedF   all   300     0.99678            

## Cell 13 — Test inference and outputs

Applies the chosen rule to the test probabilities and writes `matching_results.tsv` and `candidate_pairs.tsv`. There is exactly one row per test S1 (France included) in file order. Candidates are the exact set the model scored, and matches are a subset of them. The cell then runs read-back asserts and the official validator. In smoke mode it also writes a full-size all-empty submission to `output_smoke/full_dummy/` and validates that too.

In [26]:
def run_validator(match_path, cand_path):
    cmd = [sys.executable, 'utils/validate_submission.py', '--matching', str(match_path.relative_to(ROOT)),
           '--candidate', str(cand_path.relative_to(ROOT)), '--test-dir', 'dataset/test']
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(res.stdout[-4000:])
    if res.stderr.strip():
        print(res.stderr[-2000:])
    return res.returncode == 0 and 'PASS' in res.stdout

with section('13 test inference + write'):
    qt = pd.concat([pd.read_parquet(f, columns=['q', 't']) for f in FEAT_FILES['test']], ignore_index=True)
    q_te, t_te = qt['q'].to_numpy(np.int64), qt['t'].to_numpy(np.int64)
    del qt
    p_lgb_te, p_mlp_te = np.load(ART / 'lgb' / 'test.npy'), np.load(ART / 'mlp' / 'test.npy')
    assert len(p_lgb_te) == len(q_te) == len(p_mlp_te), (len(p_lgb_te), len(q_te), len(p_mlp_te))
    w = DECISION['w_lgb']
    p_te = ISO.predict(w * p_lgb_te + (1 - w) * p_mlp_te).astype(np.float32)
    sel_te = er_decide.apply_rule(DECISION['variant'], q_te, t_te, p_te, thr=DECISION['thr'], seed=SEED, draws=MC_DRAWS)

    rte = rec['test']
    src_te = rte['src'].to_numpy()
    assert (src_te[q_te] == 1).all() and (src_te[t_te] != 1).all(), 'pairs must be S1 x (S2 u S3)'
    NR_TE = len(rte)
    assert len(np.unique(q_te * NR_TE + t_te)) == len(q_te), 'duplicate candidate pairs'
    s1_glob = np.flatnonzero(src_te == 1)
    file_row = pd.Index(TEST_S1_IDS).get_indexer(er_text.str_take(rte, 'entity_id', s1_glob))
    assert (file_row >= 0).all()
    row_of = np.full(NR_TE, -1, np.int64)
    row_of[s1_glob] = file_row
    row = row_of[q_te]
    order = np.lexsort((t_te, row))
    row_s, t_s, sel_s = row[order], t_te[order], sel_te[order]
    ids_of = lambda idx: (lambda a, b: er_text.str_take(rte, 'entity_id', idx[a:b]))
    CAND_PATH, MATCH_PATH = OUT / 'candidate_pairs.tsv', OUT / 'matching_results.tsv'
    er_decide.write_lists(CAND_PATH, ['source1_entity_id', 'candidate_entity_ids'], TEST_S1_IDS, row_s, ids_of(t_s))
    er_decide.write_lists(MATCH_PATH, ['source1_entity_id', 'matched_entity_ids'], TEST_S1_IDS, row_s[sel_s], ids_of(t_s[sel_s]))
    print(f'  wrote {CAND_PATH.relative_to(ROOT)} ({len(t_s):,} candidate pairs) and {MATCH_PATH.relative_to(ROOT)} ({int(sel_s.sum()):,} matches)')

    # ---- read-back checks
    valid_ids = set()
    for i in (2, 3):
        valid_ids |= set(read_tsv(f'dataset/test/test_source{i}.tsv', usecols=['entity_id'])['entity_id'])
    mres = read_tsv(MATCH_PATH)
    cres_ids = read_tsv(CAND_PATH, usecols=['source1_entity_id'])['source1_entity_id']
    hdr_ok = list(mres.columns) == ['source1_entity_id', 'matched_entity_ids'] and \
             open(CAND_PATH, encoding='utf-8').readline().rstrip(chr(10)) == 'source1_entity_id' + chr(9) + 'candidate_entity_ids'
    rows_ok = (len(mres) == len(TEST_S1_IDS) == len(cres_ids) and mres['source1_entity_id'].is_unique
               and mres['source1_entity_id'].tolist() == TEST_S1_IDS and cres_ids.tolist() == TEST_S1_IDS)
    ml = mres.loc[mres['matched_entity_ids'] != '', 'matched_entity_ids'].str.split(',')
    mex = ml.explode()
    nodup_ok = not mex.reset_index().duplicated().any()
    exist_ok = bool(mex.isin(valid_ids).all()) and bool(mex.str.match('S[23]-').all())
    # matches subset of candidates: same (row, t) keys by construction; verify on the arrays
    ckeys = row_s * NR_TE + t_s
    subset_ok = bool(np.isin(ckeys[sel_s], ckeys).all())
    if SMOKE:  # small enough to re-read the candidate file fully
        cres = read_tsv(CAND_PATH)
        cl = cres['candidate_entity_ids'].str.split(',')
        both = mres['matched_entity_ids'] != ''
        subset_ok = subset_ok and all(set(m.split(',')) <= set(c) for m, c in zip(mres.loc[both, 'matched_entity_ids'], cl[both]))
        cex = cl[cres['candidate_entity_ids'] != ''].explode()
        nodup_ok = nodup_ok and not cex.reset_index().duplicated().any()
        exist_ok = exist_ok and bool(cex.isin(valid_ids).all())
    del valid_ids
    check(hdr_ok, 'exact headers')
    check(rows_ok, f'one row per test S1 in file order ({len(TEST_S1_IDS):,})')
    check(nodup_ok, 'no duplicate IDs within a list')
    check(exist_ok, 'every matched ID is an S2-/S3- ID that exists in test')
    check(subset_ok, 'matches are a subset of candidates')
    VAL_OK = check(run_validator(MATCH_PATH, CAND_PATH), f'validator on {OUT.name}/')

    dummy_ok = None
    if SMOKE:  # full-size all-empty submission: exercises the writer on every test S1
        dd = OUT / 'full_dummy'
        dd.mkdir(exist_ok=True)
        none = np.zeros(0, np.int64)
        er_decide.write_lists(dd / 'candidate_pairs.tsv', ['source1_entity_id', 'candidate_entity_ids'], TEST_S1_IDS, none, lambda a, b: [])
        er_decide.write_lists(dd / 'matching_results.tsv', ['source1_entity_id', 'matched_entity_ids'], TEST_S1_IDS, none, lambda a, b: [])
        dummy_ok = check(run_validator(dd / 'matching_results.tsv', dd / 'candidate_pairs.tsv'), 'validator on output_smoke/full_dummy/ (all-empty, every test S1)')

    ctry_te = rte['country'].astype(str).to_numpy(dtype=object)
    n_s1_c = pd.Series(ctry_te[s1_glob]).value_counts()
    matched_s1 = pd.Series(ctry_te[np.unique(q_te[sel_te])]).value_counts()
    pc_ = pd.DataFrame({'c': ctry_te[q_te], 'sel': sel_te}).groupby('c').agg(cands=('sel', 'size'), matches=('sel', 'sum'))
    pred_tab = pd.DataFrame({'country': n_s1_c.index, 'n_S1': n_s1_c.to_numpy()})
    pred_tab['cands_per_S1'] = (pred_tab['country'].map(pc_['cands']).fillna(0) / pred_tab['n_S1']).round(2)
    pred_tab['matches_per_S1'] = (pred_tab['country'].map(pc_['matches']).fillna(0) / pred_tab['n_S1']).round(3)
    pred_tab['empty_rate'] = (1 - pred_tab['country'].map(matched_s1).fillna(0) / pred_tab['n_S1']).round(4)
    print(pred_tab.to_string(index=False))
    paste(f'cell13 validator={VAL_OK} dummy={dummy_ok} headers={hdr_ok} rows={rows_ok} nodup={nodup_ok} exist={exist_ok} subset={subset_ok} pred={pred_tab.to_dict("records")}')

── 13 test inference + write ── start (RSS 0.61 GB)
  wrote output_smoke/candidate_pairs.tsv (5,462 candidate pairs) and output_smoke/matching_results.tsv (144 matches)
✅ PASS: exact headers
✅ PASS: one row per test S1 in file order (1,732,544)
✅ PASS: no duplicate IDs within a list
✅ PASS: every matched ID is an S2-/S3- ID that exists in test
✅ PASS: matches are a subset of candidates
ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (1732488 empty, 56 non-empty).
  candidate_pairs.tsv: 1732544 rows (1732343 empty, 201 non-empty).

PASS — no blocking issues found. Safe to submit.

✅ PASS: validator on output_smoke/
ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (1732544 empty, 0 non-empty).
  candidate_pairs.tsv: 1732544 rows (1732544 empty, 0 non-empty).

PASS — no blocking issues found. Safe to submit.

✅ PASS: v

## Cell 14 — Summary

Results table: CV macro-F0.5 per model and per decision variant, leave-country-out scores, blocking recall, and wall time plus peak RSS per section. It is also written to `reports/summary.md` (`reports/summary_smoke.md` in smoke mode).

In [27]:
def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(map(str, cols)) + ' |', '|' + '---|' * len(cols)]
    out += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.itertuples(index=False)]
    return chr(10).join(out)

with section('14 summary'):
    cv_tab = pd.DataFrame([{'model': k, 'oof_macroF05_best_threshold': round(float(v), 5)} for k, v in CV_SCORES.items()])
    dec_tab = var_tab.round(5)
    loco_tab = (pd.concat(list(LOCO_RES.values()), ignore_index=True) if any(len(v) for v in LOCO_RES.values())
                else pd.DataFrame({'note': ['only one training country']}))
    blk_tab = pd.DataFrame([{'union_pair_recall': round(BLOCK_RECALL, 5), 's1_full_recall': round(BLOCK_FULL_S1, 5),
                             'cands_per_S1_train': round(BLOCK_CANDS, 1), 'retrievers': 'char-tfidf + word-tfidf + dense(MLX) + rare/postal'}])
    test_tab = pd.DataFrame([{'group': g, **v} for g, v in test_stats.items()])
    sec_tab = pd.DataFrame(SECTION_LOG)
    mode = 'SMOKE' if SMOKE else 'FULL'
    variant, w_final, thr_final, f_final = DECISION['variant'], DECISION['w_lgb'], DECISION['thr'], DECISION['oof_macro_f05']
    parts = ['# Entity resolution run summary', '',
             f'Mode: {mode}. Embeddings: {EMB_REPO} ({EMB_POOLING}+L2, fp16, PCA {EMB_DIM}), fields {list(EMB_FIELDS)}. MLP backend: {BACKEND}.',
             f'Decisions: ONE_TO_ONE={ONE_TO_ONE}, SAME_COUNTRY={SAME_COUNTRY}. Final rule: {variant} (w_lgb={w_final}, thr={thr_final}), OOF macro-F0.5 {f_final:.5f}.', '',
             '## CV macro-F0.5 per model (OOF, best global threshold)', md_table(cv_tab), '',
             '## Decision variants (OOF, cross-fitted isotonic)', md_table(dec_tab), '',
             '## Leave-one-country-out (France proxy; raw probabilities)', md_table(loco_tab), '',
             '## Blocking (train sample)', md_table(blk_tab), '', '## Test candidates', md_table(test_tab), '',
             '## Test predictions', md_table(pred_tab), '',
             '## Wall time and memory per section', md_table(sec_tab), '',
             'No external data was used: no geocoding, APIs, registries, web lookups or external datasets; only pretrained bge-m3 weights and hand-written dictionaries.']
    summary_md = chr(10).join(parts)
    SUMMARY_PATH = REP / ('summary_smoke.md' if SMOKE else 'summary.md')
    SUMMARY_PATH.write_text(summary_md, encoding='utf-8')
    print(summary_md)
    check(SUMMARY_PATH.exists(), f'summary written to {SUMMARY_PATH.relative_to(ROOT)}')
    peak = max(r['peak_rss_gb'] for r in SECTION_LOG)
    total_h = sum(r['wall_s'] for r in SECTION_LOG) / 3600
    cv_short = {k: round(float(v), 5) for k, v in CV_SCORES.items()}
    paste(f'cell14 cv={cv_short} final={variant} oofF05={f_final:.5f} block_recall={BLOCK_RECALL:.4f} peak_rss={peak:.2f}GB total={total_h:.2f}h validator={VAL_OK}')

── 14 summary ── start (RSS 2.15 GB)
# Entity resolution run summary

Mode: SMOKE. Embeddings: mlx-community/bge-m3-mlx-fp16 (cls+L2, fp16, PCA 256), fields ['name', 'addr', 'combo']. MLP backend: mlx.
Decisions: ONE_TO_ONE=True, SAME_COUNTRY=True. Final rule: threshold (w_lgb=0.3, thr=0.75), OOF macro-F0.5 0.99678.

## CV macro-F0.5 per model (OOF, best global threshold)
| model | oof_macroF05_best_threshold |
|---|---|
| lgb | 0.99551 |
| mlp | 0.99635 |
| blend | 0.99697 |

## Decision variants (OOF, cross-fitted isotonic)
| variant | group | n_s1 | macro_F0.5 | singleton_acc | nonsingleton_F0.5 | threshold |
|---|---|---|---|---|---|---|
| threshold | all | 300 | 0.99678 | 1.0 | 0.99661 | 0.75 |
| threshold | India | 120 | 0.99548 | 1.0 | 0.99524 | 0.75 |
| threshold | US | 180 | 0.99765 | 1.0 | 0.99753 | 0.75 |
| threshold+assign | all | 300 | 0.99678 | 1.0 | 0.99661 | 0.75 |
| threshold+assign | India | 120 | 0.99548 | 1.0 | 0.99524 | 0.75 |
| threshold+assign | US | 180 | 0.9976

## Methodology, models and licenses

**Models** (≤ 8B parameters, MIT / Apache-2.0 only). *Please confirm each license on its Hugging Face card before submitting.*

| Role | Repo actually loaded | License | Original base model | License | Params |
|---|---|---|---|---|---|
| Dense embeddings (cells 1a, 1b, 6) | `mlx-community/bge-m3-mlx-fp16` (MLX fp16 conversion) | MIT (as on its card) | `BAAI/bge-m3` | MIT | ~568M |
| Parity reference / `USE_ST_MPS` fallback | `BAAI/bge-m3` via sentence-transformers | MIT | — | — | ~568M |

LightGBM and the Siamese MLP are trained from scratch on the provided training data.

**Software** (all permissive): MLX (MIT), numpy / pandas / scipy / scikit-learn / joblib / psutil (BSD), pyarrow (Apache-2.0), rapidfuzz (MIT), LightGBM (MIT), PyTorch (BSD-3), sentence-transformers / transformers / tokenizers / safetensors / huggingface_hub (Apache-2.0). `mlx-embeddings` (GPL-3.0) and `unidecode` (GPL) are **not** used: the XLM-R encoder is written in `mlx.nn` (`er_mlx_encoder.py`), and accents are stripped with `unicodedata` NFKD.

**Apple-silicon stack.** bge-m3 runs in MLX in fp16 on the M1 Pro GPU (CLS pooling + L2, parity-checked against sentence-transformers). Vectors are PCA-projected to `EMB_DIM` and stored as float16 memmaps. Dense top-k search runs in MLX with chunked `matmul` + `argpartition`, sized to the pool so the score matrix stays under ~300 MB. The MLP trains in MLX in fp32 with a compiled `value_and_grad` step. Everything else runs on the CPU performance cores (rapidfuzz `cpdist`, sparse TF-IDF, LightGBM).

**Blocking.** For each S1, candidates are the union over S2 ∪ S3 of the same country string (open set) from four retrievers: char 2–4 TF-IDF on core names, word TF-IDF on name+address, dense cosine, and rare-token/postal keys. Every retriever's exact score is then computed for every union pair.

**Features.** String similarities (ratio, partial, token-sort, token-set, Jaro-Winkler, Jaccard) on raw-normalised names, suffix-stripped names and addresses. Also dense cosines, legal-suffix and postal / street-number agreement, digits, landmark flags and acronyms. Context features: ranks, reverse ranks, mutual-best, score gaps, candidate counts and cross-source corroboration. Country is never a feature.

**Decision.** The LightGBM and MLP probabilities are blended with a weight chosen on out-of-fold macro-F0.5, then isotonic-calibrated. When `ONE_TO_ONE`, each S2/S3 record is kept only for its most probable S1. Each S1's output set is then chosen from {global threshold, threshold+assignment, expected-F0.5, expected-F0.5+assignment}; the variant with the best OOF macro-F0.5 wins. Expected-F0.5 picks the top-k that maximises Monte-Carlo expected F0.5, and k = 0 is allowed. Nothing is tuned on test.

**No external data.** No geocoding, APIs, registries, web lookups or external datasets. The only non-code inputs are pretrained bge-m3 weights and hand-written dictionaries (legal suffixes, address abbreviations, landmark words, a Brahmic-script transliteration table).